# Duplicate Detection Using 3-grams and Jaccard Similarity

In [ ]:
import re

## 1. Sample Document Creation

In [23]:
# 10 sample documents with intentional near-duplicates
documents = {
    "doc1": "Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions from data without being explicitly programmed.",
    
    "doc2": "Machine learning is a branch of artificial intelligence that allows computers to learn and make decisions from data without explicit programming.",
    
    "doc3": "Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions from data without being explicitly programmed for tasks.",
    
    "doc4": "Climate change refers to long-term shifts in global temperatures and weather patterns, primarily caused by human activities and greenhouse gas emissions.",
    
    "doc5": "Photosynthesis is the process by which plants convert sunlight, carbon dioxide, and water into glucose and oxygen using chlorophyll.",
    
    "doc6": "Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions through data without being programmed.",
    
    "doc7": "Quantum computing uses quantum mechanical phenomena like superposition and entanglement to perform calculations exponentially faster than classical computers.",
    
    "doc8": "The human brain contains approximately 86 billion neurons that communicate through electrical and chemical signals to process information.",
    
    "doc9": "Renewable energy sources include solar, wind, hydroelectric, and geothermal power, which are sustainable alternatives to fossil fuels.",
    
    "doc10": "DNA, or deoxyribonucleic acid, is the hereditary material in humans and most organisms that contains genetic instructions for development and function."
}

print(f"Created {len(documents)} documents")
print("\nDocument lengths:")
for doc_id, content in documents.items():
    print(f"{doc_id}: {len(content)} characters")

Created 10 documents

Document lengths:
doc1: 153 characters
doc2: 145 characters
doc3: 163 characters
doc4: 153 characters
doc5: 132 characters
doc6: 145 characters
doc7: 158 characters
doc8: 138 characters
doc9: 134 characters
doc10: 151 characters


## 2. 3-grams (Word Shingles) Generation

In [24]:
def generate_word_shingles(text, k=3):
    # Convert to lowercase and split into words
    words = re.findall(r'\b\w+\b', text.lower())
    
    shingles = set()
    for i in range(len(words) - k + 1):
        shingle = ' '.join(words[i:i+k])
        shingles.add(shingle)
    
    return shingles

# Generating 3-grams for all documents
doc_shingles = {}
for doc_id, content in documents.items():
    shingles = generate_word_shingles(content, k=3)
    doc_shingles[doc_id] = shingles
    print(f"{doc_id}: {len(shingles)} unique 3-grams")

# Example 3-grams from first document
print(f"\nExample 3-grams from doc1:")
for i, shingle in enumerate(list(doc_shingles['doc1'])[:5]):
    print(f"  {i+1}. '{shingle}'")

doc1: 20 unique 3-grams
doc2: 19 unique 3-grams
doc3: 22 unique 3-grams
doc4: 20 unique 3-grams
doc5: 17 unique 3-grams
doc6: 19 unique 3-grams
doc7: 16 unique 3-grams
doc8: 16 unique 3-grams
doc9: 15 unique 3-grams
doc10: 19 unique 3-grams

Example 3-grams from doc1:
  1. 'machine learning is'
  2. 'being explicitly programmed'
  3. 'artificial intelligence that'
  4. 'that enables computers'
  5. 'a subset of'


## 3. Jaccard Similarity Calculation

In [25]:
def jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    if union == 0:
        return 0
    
    return intersection / union

# Calculating similarities for all pairs
similarities = {}
doc_ids = list(documents.keys())

print("Jaccard Similarity Matrix:")
print("\t" + "\t".join(doc_ids))

for i, doc1 in enumerate(doc_ids):
    row = [doc1]
    for j, doc2 in enumerate(doc_ids):
        if i <= j:  # upper triangle + diagonal
            similarity = jaccard_similarity(doc_shingles[doc1], doc_shingles[doc2])
            similarities[(doc1, doc2)] = similarity
            row.append(f"{similarity:.3f}")
        else: # lower triangle
            similarity = similarities[(doc2, doc1)]
            row.append(f"{similarity:.3f}")
    
    print("\t".join(row))

Jaccard Similarity Matrix:
	doc1	doc2	doc3	doc4	doc5	doc6	doc7	doc8	doc9	doc10
doc1	1.000	0.393	0.909	0.000	0.000	0.625	0.000	0.000	0.000	0.000
doc2	0.393	1.000	0.367	0.000	0.000	0.267	0.000	0.000	0.000	0.000
doc3	0.909	0.367	1.000	0.000	0.000	0.577	0.000	0.000	0.000	0.000
doc4	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000
doc5	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000
doc6	0.625	0.267	0.577	0.000	0.000	1.000	0.000	0.000	0.000	0.000
doc7	0.000	0.000	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000
doc8	0.000	0.000	0.000	0.000	0.000	0.000	0.000	1.000	0.000	0.000
doc9	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	1.000	0.000
doc10	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	1.000


## 4. Near-Duplicates Identification

In [26]:
threshold = 0.6
near_duplicates = []

# Pairs with similarity >= threshold
for (doc1, doc2), similarity in similarities.items():
    if doc1 != doc2 and similarity >= threshold:
        near_duplicates.append((doc1, doc2, similarity))

print(f"Near-duplicate pairs (similarity ≥ {threshold}):")
print("=" * 50)

if near_duplicates:
    for doc1, doc2, similarity in near_duplicates:
        print(f"\n{doc1} - {doc2}: {similarity:.3f}")
        print(f"Content of {doc1}:")
        print(f"  {documents[doc1]}")
        print(f"Content of {doc2}:")
        print(f"  {documents[doc2]}")
        
        # Analyze if it's a true duplicate
        if similarity > 0.8:
            print("  → TRUE DUPLICATE: Very high similarity, likely same content with minor variations")
        elif similarity > 0.6:
            print("  → NEAR DUPLICATE: Moderate similarity, possibly related content")
        
        # Show common 3-grams
        common_shingles = doc_shingles[doc1].intersection(doc_shingles[doc2])
        print(f"  Common 3-grams ({len(common_shingles)}): {list(common_shingles)[:3]}...")
else:
    print("No near-duplicates found with the current threshold.")

print(f"\nTotal pairs analyzed: {len(similarities)}")
print(f"Near-duplicate pairs found: {len(near_duplicates)}")

Near-duplicate pairs (similarity ≥ 0.6):

doc1 - doc3: 0.909
Content of doc1:
  Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions from data without being explicitly programmed.
Content of doc3:
  Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions from data without being explicitly programmed for tasks.
  → TRUE DUPLICATE: Very high similarity, likely same content with minor variations
  Common 3-grams (20): ['machine learning is', 'being explicitly programmed', 'artificial intelligence that']...

doc1 - doc6: 0.625
Content of doc1:
  Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions from data without being explicitly programmed.
Content of doc6:
  Machine learning is a subset of artificial intelligence that enables computers to learn and make decisions through data without being programmed.
  → NEAR DUPLICATE: Moderate

## 5. Analysis: Why LSH + MinHash is Needed for Web-Scale

The current approach works well for small collections but has limitations for web-scale applications.

In [27]:
# Demonstrate computational complexity
n_docs = len(documents)
n_comparisons = (n_docs * (n_docs - 1)) // 2

print("Computational Analysis:")
print(f"Documents: {n_docs}")
print(f"Pairwise comparisons needed: {n_comparisons}")
print(f"Time complexity: O(n²)")

# Extrapolate to web scale
web_scale_docs = [1000, 10000, 100000, 1000000, 10000000]
print("\nWeb-scale extrapolation:")
for n in web_scale_docs:
    comparisons = (n * (n - 1)) // 2
    print(f"{n:,} documents → {comparisons:,} comparisons")

Computational Analysis:
Documents: 10
Pairwise comparisons needed: 45
Time complexity: O(n²)

Web-scale extrapolation:
1,000 documents → 499,500 comparisons
10,000 documents → 49,995,000 comparisons
100,000 documents → 4,999,950,000 comparisons
1,000,000 documents → 499,999,500,000 comparisons
10,000,000 documents → 49,999,995,000,000 comparisons
